# Scorecard diagnostics and characteristic presentations

Extend the from-scratch scorecard with VIF, coefficient inference, fixed-bin PSI, policy flags, and an editable characteristic-review presentation when the book dependency is installed.

All data are generated locally unless this notebook explicitly calls a reviewed adapter. Results are educational and require independent validation before any real use.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from creditriskbook.data.datasets import load_dataset
from creditriskbook.models import split_dataset
from creditriskbook.scorecard import (
    LogisticScorecard, binned_population_stability, coefficient_inference,
    export_characteristic_presentation, scorecard_policy_flags,
    variance_inflation_factors,
)

bundle = load_dataset("synthetic_retail", n_rows=4_000, seed=1111)
train, test = split_dataset(bundle, bundle.frame, seed=1111)
features = ["income", "employment_years", "debt_to_income", "utilisation",
            "enquiries_6m", "loan_amount", "product", "home_ownership"]
scorecard = LogisticScorecard().fit(train[features], train[bundle.target])

In [ ]:
reference_bins = scorecard.binning.transform(train[features])
current_bins = scorecard.binning.transform(test[features])
detail, stability = binned_population_stability(reference_bins, current_bins)
woe_reference = scorecard.encoder_.transform(reference_bins).astype(float)
vif = variance_inflation_factors(woe_reference)
inference = coefficient_inference(scorecard)
flags = scorecard_policy_flags(scorecard, minimum_bin_count=20)
assert set(stability["feature"]) == set(features)
print(stability)
print(vif)
print(inference)
print(flags)

In [ ]:
try:
    import pptx  # noqa: F401
except ImportError:
    print("Install the 'book' extra to generate PowerPoint.")
else:
    with TemporaryDirectory() as directory:
        path = export_characteristic_presentation(
            scorecard, Path(directory) / "characteristic_review.pptx"
        )
        assert path.exists() and path.stat().st_size > 10_000
        print("Generated", path.name, path.stat().st_size, "bytes")

Diagnostics are evidence, not automatic approval thresholds. Review the business definition, availability, stability, sign, sparse bins, missing and unseen behavior, policy relevance, and legal use for every characteristic.